In [ ]:
# ════════════════════════════════════════════════════════════════════════════════════════════
#  FMA-v4 — TABLE III (Flickr30K) RUNNER, steps 1 → 2 → 3
#
#  STAGE 1  Reproduce the SA-AET baseline: CLIP_ViT source, seed 42, FULL test set (1000 images /
#           5000 captions). Prints a check against the paper's Table III row. Do not go to stage 2
#           until every transfer cell is within ±2 points of the paper (or you understand why not).
#  STAGE 2  Full FMA-v4 on CLIP_ViT with seeds 42/43/44, plus SA-AET seeds 43/44 (paired seeds).
#  STAGE 3  Remaining Table III rows: ALBEF and TCL sources (SA-AET + FMA-v4), CLIP_CNN source
#           (SA-AET only: FMA hooks need ViT blocks, so the CLIP_CNN row is reported as N/A for ours).

#  SELF-CONTAINED: works on a fresh Colab runtime (clones repo, builds env, data, eval_reg.py, reg_attack.py).
# ════════════════════════════════════════════════════════════════════════════════════════════
import csv, datetime, glob, hashlib, json, os, re, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path
import numpy as np

# ── SETTINGS ────────────────────────────────────────────────────────────────────────────────
STAGE = 3                       # 1 = reproduce SA-AET | 2 = + FMA-v4 on CLIP_ViT, 3 seeds | 3 = + ALBEF/TCL/CLIP_CNN rows
N_IMAGES = 1000                 # Table III needs the FULL split (1000). Smaller values are for smoke tests only.
SEEDS = [42, 43]            # stage 2 seeds (CLIP_ViT row)
STAGE3_SEEDS = [42, 43]     # stage 3 seeds; use [42] for a faster first draft, add 43/44 later (cached runs are kept)
STAGE3_SOURCES = ["TCL"]
SOURCE_ITM = False               # True: ALBEF/TCL *source* column ranked with ITM re-ranking, as in the paper.
                                # False: feature-similarity ranking (faster, but the white-box cell is not paper-comparable)
BATCH_SIZE = 8                  # paper/repo default. Keep identical for every method (it changes the in-batch loss).
SAVE_IMAGES = False
RUN_ABLATIONS = False           # step 8 only; leave False for Table III
REPRO_TOL = 2.0                 # stage-1 check: |reproduced − paper| per transfer cell, in ASR points

LAM, LAYERS = 4.0, "7,8,9,10"
FMA_VER = "fma-v4"

# Frozen FMA-v4 hyper-parameters (unchanged from the sub400 ablation; do NOT retune on the test split)
FMA_CFG = {
    "act_power": 1.0,           # MAD z-score band-pass filter strength (0.0 = uniform)
    "reg_z": 3.0,               # suppresses high-norm ViT register outliers with robust z-score > 3.0
    "bg_z": -0.8,               # down-weights low-information background patches with z-score < -0.8
    "patch2x2_beta": 0.25,      # 2x2 stride-2 pyramid pooled term
    "patch4x4_beta": 0.20,      # 4x4 stride-2 regional pooled term
    "pool_stride": 2,           # stride=2 preserves sharp spatial edges
    "cls_beta": 0.0,            # 0.0 = spatial-only alignment (no intermediate CLS interference)
    "focal_alpha": 2.5,         # self-paced focus on stubborn unperturbed tokens
    "focal_margin": 0.15,       # margin between broken & unbroken tokens
    "layer_focus": 0.8          # cross-layer focal weighting
}

MODELS = ["ALBEF", "TCL", "CLIP_ViT", "CLIP_CNN"]
VIT_SOURCES = {"ALBEF", "TCL", "CLIP_ViT"}      # FMA hooks ViT blocks; CLIP_CNN (ResNet-101) has none
ITM_MODELS = {"ALBEF", "TCL"}

SAAET_LABEL = "SA-AET"
FMA_LABEL   = f"{FMA_VER}_L7-10_lam{LAM:g}"
TOK_LABEL   = f"tok-v2_L7-10_lam{LAM:g}"

DISPLAY = {
    SAAET_LABEL: "SA-AET baseline",
    FMA_LABEL:   "Full FMA-v4 (ours)",
    TOK_LABEL:   "per-token cosine"
}

ABLATIONS = {
    f"{FMA_VER}-noms_L7-10_lam{LAM:g}": (
        "w/o multi-scale", "no 2x2 / 4x4 pyramid pooling",
        dict(FMA_CFG, patch2x2_beta=0.0, patch4x4_beta=0.0)
    ),
    f"{FMA_VER}-nofocal_L7-10_lam{LAM:g}": (
        "w/o focal", "no token or layer focal weighting",
        dict(FMA_CFG, focal_alpha=0.0, layer_focus=0.0)
    ),
    f"{FMA_VER}-noact_L7-10_lam{LAM:g}": (
        "w/o band-pass wt", "uniform spatial weights (act_power=0)",
        dict(FMA_CFG, act_power=0.0, reg_z=99.0)
    ),
    f"{FMA_VER}-withcls_L7-10_lam{LAM:g}": (
        "+ CLS term", "adds intermediate CLS (cls_beta=0.15)",
        dict(FMA_CFG, cls_beta=0.15)
    ),
    TOK_LABEL: (
        "per-token cosine", "uniform token cosine (no MS, focal, band-pass)",
        dict(FMA_CFG, act_power=0.0, reg_z=99.0, patch2x2_beta=0.0, patch4x4_beta=0.0,
             cls_beta=0.0, focal_alpha=0.0, layer_focus=0.0)
    ),
}

LABEL_CFG = {FMA_LABEL: FMA_CFG, **{lab: cfg for lab, (_, _, cfg) in ABLATIONS.items()}}
for lab, (name, _, _) in ABLATIONS.items():
    DISPLAY[lab] = name

# Paper Table III (Flickr30K), R@1 ASR (%) as (TR, IR). Source -> attack -> target.
PAPER_T3 = {
    "ALBEF": {
        "SGA":    {"ALBEF": (99.90, 99.98), "TCL": (87.88, 88.05), "CLIP_ViT": (36.69, 46.78), "CLIP_CNN": (39.59, 49.78)},
        "DRA":    {"ALBEF": (99.90, 99.93), "TCL": (91.57, 91.17), "CLIP_ViT": (46.26, 56.80), "CLIP_CNN": (49.55, 59.01)},
        "SA-AET": {"ALBEF": (99.90, 99.98), "TCL": (96.42, 96.02), "CLIP_ViT": (55.58, 63.89), "CLIP_CNN": (57.22, 65.59)},
    },
    "TCL": {
        "SGA":    {"ALBEF": (92.49, 92.77), "TCL": (100.0, 100.0), "CLIP_ViT": (36.81, 46.97), "CLIP_CNN": (41.89, 51.53)},
        "DRA":    {"ALBEF": (95.20, 95.58), "TCL": (100.0, 99.98), "CLIP_ViT": (47.24, 57.28), "CLIP_CNN": (52.23, 62.23)},
        "SA-AET": {"ALBEF": (98.85, 98.50), "TCL": (100.0, 100.0), "CLIP_ViT": (56.20, 63.47), "CLIP_CNN": (59.77, 67.86)},
    },
    "CLIP_ViT": {
        "SGA":    {"ALBEF": (22.42, 34.59), "TCL": (25.08, 36.45), "CLIP_ViT": (100.0, 100.0), "CLIP_CNN": (53.26, 61.10)},
        "DRA":    {"ALBEF": (27.84, 42.84), "TCL": (27.82, 44.60), "CLIP_ViT": (100.0, 100.0), "CLIP_CNN": (64.88, 69.50)},
        "SA-AET": {"ALBEF": (36.60, 50.44), "TCL": (39.20, 51.10), "CLIP_ViT": (100.0, 100.0), "CLIP_CNN": (71.01, 74.10)},
    },
    "CLIP_CNN": {
        "SGA":    {"ALBEF": (15.64, 28.60), "TCL": (18.02, 33.07), "CLIP_ViT": (39.02, 51.45), "CLIP_CNN": (99.87, 99.90)},
        "DRA":    {"ALBEF": (19.50, 34.59), "TCL": (21.60, 37.88), "CLIP_ViT": (48.47, 59.12), "CLIP_CNN": (99.87, 99.90)},
        "SA-AET": {"ALBEF": (23.98, 38.28), "TCL": (27.29, 41.81), "CLIP_ViT": (54.11, 64.21), "CLIP_CNN": (100.0, 99.97)},
    },
}

REPO = Path("/content/SA-AET")
ENV_PY = "/content/sa_env/bin/python"
OUT = Path("/content/drive/MyDrive/sa_aet_phase1")
FLICKR_ROOT = Path("/content/drive/MyDrive/datasets/flickr30k")
FLICKR_CACHE = Path("/content/drive/MyDrive/datasets/flickr30k_test_images")
FLICKR_ZIP_URL = "https://huggingface.co/datasets/nlphuji/flickr30k/resolve/main/flickr30k-images.zip"
FLICKR_ZIP = Path("/content/flickr30k-images.zip")
LOCAL_ROOT = Path("/content/flickr_local")
FOUT = OUT / "flickr30k"


def sh(cmd, log=None):
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    f = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="")
        if f:
            f.write(line); f.flush()
    p.wait()
    if f:
        f.close()
    if p.returncode != 0:
        raise SystemExit(f"command failed (exit {p.returncode}): {cmd}")


# ── attack module: FMA-v4 (MAD z-score band-pass + stride-2 pyramid pooling) — UNCHANGED ─────
REG_ATTACK = r'''
import json, math, os
import torch
import torch.nn.functional as F
from torchvision import transforms
from SA_AET import ImageAttacker

SA_SCALES = [0.5, 0.75, 1.25, 1.5]
FMA_DEFAULTS = {"act_power": 1.0, "reg_z": 3.0, "bg_z": -0.8,
                "patch2x2_beta": 0.25, "patch4x4_beta": 0.20, "pool_stride": 2,
                "cls_beta": 0.0, "focal_alpha": 2.5, "focal_margin": 0.15, "layer_focus": 0.8}


def prefix_len(T):
    g = int(math.isqrt(T))
    while g > 0 and T - g * g > 8:
        g -= 1
    p = T - g * g
    return p if g > 0 and 0 <= p <= 8 else None


def _focal(base, cos, margin, alpha):
    with torch.no_grad():
        if alpha <= 0.0:
            return base / (base.sum(dim=1, keepdim=True) + 1e-12)
        # Emphasize stubborn tokens/regions whose cosine similarity to clean is still high
        f = base * (cos.detach() + margin).clamp(min=0.02) ** alpha
        return f / (f.sum(dim=1, keepdim=True) + 1e-12)


@torch.no_grad()
def fma_precompute(a, act_power=1.0, reg_z=3.0, bg_z=-0.8, pool_stride=2, **_):
    """
    Robust z-score band-pass spatial weighting + Salience-weighted Pyramid Pooling:
      1. Computes robust z-score per token: z = (norm - median) / (1.4826 * MAD)
      2. Band-pass filter:
         - High-pass: sigmoid((z - bg_z) * 1.5) down-weights flat background patches
         - Low-pass:  exp(-0.5 * relu(z - reg_z)^2) suppresses high-norm ViT register outliers
      3. Stride-2 Pyramid Pooling: matches CNN 2x spatial downsampling without box-blurring edges
    """
    a = a.float()
    B, T, D = a.shape
    p = prefix_len(T)
    g = int(math.isqrt(T - p)) if p is not None else 0

    a_sp = a[:, p:] if (p is not None and p > 0) else a
    n = a_sp.norm(dim=-1)
    med = n.median(dim=1, keepdim=True).values
    mad = (n - med).abs().median(dim=1, keepdim=True).values * 1.4826 + 1e-6
    z = (n - med) / mad

    if act_power > 0.0:
        w_fg = torch.sigmoid((z - bg_z) * 1.5)                  # suppress flat background (z < bg_z)
        w_reg = torch.exp(-0.5 * torch.relu(z - reg_z) ** 2)    # suppress register outliers (z > reg_z)
        w_sp = (w_fg * w_reg) ** act_power
    else:
        w_sp = torch.ones_like(n)
    w_sp = w_sp / (w_sp.sum(dim=1, keepdim=True) + 1e-12)

    c = {"a": a, "a_sp": a_sp, "w_sp": w_sp, "p": p, "g": g, "s": int(pool_stride)}
    if p is not None and g >= 6 and p + g * g == T:
        s = int(pool_stride)
        # Weight unit tokens by sqrt(w_sp) inside each pooling window so salient edges dominate background
        sw = w_sp.sqrt().unsqueeze(-1)
        u_sp = (sw * F.normalize(a_sp, dim=-1)).transpose(1, 2).reshape(B, D, g, g)
        w_grid = w_sp.view(B, 1, g, g)
        W2 = F.avg_pool2d(w_grid, 2, stride=s).flatten(1)
        W4 = F.avg_pool2d(w_grid, 4, stride=s).flatten(1)
        c.update({
            "sw": sw,
            "A2": F.avg_pool2d(u_sp, 2, stride=s).flatten(2),
            "W2": W2 / (W2.sum(1, keepdim=True) + 1e-12),
            "A4": F.avg_pool2d(u_sp, 4, stride=s).flatten(2),
            "W4": W4 / (W4.sum(1, keepdim=True) + 1e-12),
        })
    return c


def fma_similarity(c, b, patch2x2_beta=0.25, patch4x4_beta=0.20, cls_beta=0.0,
                   focal_alpha=2.5, focal_margin=0.15, **_):
    b = b.float()
    B, D, p, g, s = b.shape[0], b.shape[-1], c["p"], c["g"], c["s"]
    b_sp = b[:, p:] if (p is not None and p > 0) else b

    tok_sp = F.cosine_similarity(c["a_sp"], b_sp, dim=-1)
    tok_s = (_focal(c["w_sp"], tok_sp, focal_margin, focal_alpha) * tok_sp).sum(1)

    if "A2" not in c or (patch2x2_beta == 0.0 and patch4x4_beta == 0.0):
        if p > 0 and cls_beta > 0:
            cls_sim = F.cosine_similarity(c["a"][:, :p], b[:, :p], dim=-1).mean(1)
            return (1.0 - cls_beta) * tok_s + cls_beta * cls_sim
        return tok_s

    ub = (c["sw"] * F.normalize(b_sp, dim=-1)).transpose(1, 2).reshape(B, D, g, g)
    p2 = F.cosine_similarity(c["A2"], F.avg_pool2d(ub, 2, stride=s).flatten(2), dim=1)
    p4 = F.cosine_similarity(c["A4"], F.avg_pool2d(ub, 4, stride=s).flatten(2), dim=1)
    p2_s = (_focal(c["W2"], p2, focal_margin, focal_alpha) * p2).sum(1)
    p4_s = (_focal(c["W4"], p4, focal_margin, focal_alpha) * p4).sum(1)

    if p > 0 and cls_beta > 0:
        cls_sim = F.cosine_similarity(c["a"][:, :p], b[:, :p], dim=-1).mean(1)
        return ((1.0 - patch2x2_beta - patch4x4_beta - cls_beta) * tok_s
                + patch2x2_beta * p2_s + patch4x4_beta * p4_s + cls_beta * cls_sim)
    return (1.0 - patch2x2_beta - patch4x4_beta) * tok_s + patch2x2_beta * p2_s + patch4x4_beta * p4_s


def vision_blocks(model):
    if hasattr(model, "visual") and hasattr(model.visual, "transformer"):
        return model.visual.transformer.resblocks, True
    if hasattr(model, "visual_encoder"):
        return model.visual_encoder.blocks, False
    raise SystemExit("this source model has no ViT blocks (CLIP_CNN): use --ifd_mode none")


class RegImageAttacker(ImageAttacker):
    def __init__(self, model, normalization, mode, layers, lam, basis=None, **kw):
        super().__init__(normalization, **kw)
        self.mode, self.layers, self.lam = mode, list(layers), float(lam)
        self.cfg = dict(FMA_DEFAULTS)
        if mode == "fma" and os.environ.get("FMA_CFG"):
            self.cfg.update(json.loads(os.environ["FMA_CFG"]))
        self.w = torch.full((len(self.layers),), 1.0 / len(self.layers))
        blocks, self.lnd = vision_blocks(model)
        self.feats, self.slice_i, self.b, self.cache = {}, 0, None, None
        self.handles = [blocks[l].register_forward_hook(self._hook(l)) for l in self.layers]
        print(f"  RegImageAttacker: mode={mode}, layers={[l + 1 for l in self.layers]}, lam={self.lam}, cfg={self.cfg}")

    def close(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def _hook(self, l):
        def fn(module, inputs, output):
            t = output[0] if isinstance(output, tuple) else output
            self.feats[l] = t.permute(1, 0, 2) if self.lnd else t
            self.slice_i = 0
        return fn

    def _ifd_fma(self, rows):
        sims = []
        for l in self.layers:
            c = {k: (v[rows] if torch.is_tensor(v) else v) for k, v in self.cache[l].items()}
            sims.append(fma_similarity(c, self.feats[l][rows], **self.cfg))
        S = torch.stack(sims)
        w = self.w.to(S.device)[:, None]
        if self.cfg["layer_focus"] > 0:
            w = w * (S.detach() + self.cfg["focal_margin"]).clamp(min=0.02) ** self.cfg["layer_focus"]
        w = w / (w.sum(dim=0, keepdim=True) + 1e-12)
        return (w * S).sum(0).mean()

    def _clean_views(self, imgs):
        H, W = imgs.shape[-2:]
        back = transforms.Resize((H, W), interpolation=transforms.InterpolationMode.BICUBIC)
        out = [imgs]
        for r in SA_SCALES:
            down = transforms.Resize((int(r * H), int(r * W)), interpolation=transforms.InterpolationMode.BICUBIC)
            out.append(back(torch.clamp(down(imgs), 0.0, 1.0)))
        return torch.cat(out, 0)

    def txt_guided_attack(self, model, imgs, txt2img, all_txt_supervisions, device, scales=None, txt_embeds=None):
        self.b = imgs.shape[0]
        with torch.no_grad():
            model.inference_image(self.normalization(self._clean_views(imgs)))
            self.ref = {l: self.feats[l].detach().float().clone() for l in self.layers}
            if self.mode == "fma":
                self.cache = {l: fma_precompute(self.ref[l], **self.cfg) for l in self.layers}
        return super().txt_guided_attack(model, imgs, txt2img, all_txt_supervisions, device,
                                         scales=scales, txt_embeds=txt_embeds)

    def loss_func(self, adv_imgs_embeds, txts_embeds, txt2img, all_txt_supervisions):
        loss = super().loss_func(adv_imgs_embeds, txts_embeds, txt2img, all_txt_supervisions)
        if self.b is None or not self.feats:
            return loss
        n, b = next(iter(self.feats.values())).shape[0], self.b
        if n == b:
            rows = slice(0, b)
        elif n == 5 * b:
            i = self.slice_i % 5
            self.slice_i += 1
            rows = slice(i * b, (i + 1) * b)
        else:
            return loss
        return loss - self.lam * self._ifd_fma(rows)
'''

# ── evaluation driver: eval_reg.py — UNCHANGED (written into the repo every run) ─────────────
EVAL_REG = r'''"""eval_reg.py - SA-AET evaluation driver with optional FMA intermediate-feature regulariser.

Reuses eval_AET.py (model loading, retrieval_eval, ITM re-ranking) and adds:
  --ifd_mode {none,fma}   none = plain SA-AET baseline, fma = SA-AET + FMA loss (reg_attack.py)
  --ifd_layers 7,8,9,10   1-based ViT block indices hooked for the FMA loss
  --lam 4.0               weight of the FMA term
Environment variables:
  PERIMG_DIR=<dir>        save per-image / per-caption ranks + adversarial captions
  ADV_DIR=<dir>           save adversarial images as PNG
  SKIP_SOURCE_ITM=1       source ALBEF/TCL: rank with feature similarity, skip slow ITM re-ranking
  FMA_CFG='<json>'        FMA hyper-parameters (read by reg_attack.py)
  FAST_ITM=1 (default)    batched ALBEF/TCL ITM re-ranking (same scores, fewer forwards); 0 = original loop
  ITM_CHUNK=4             queries per batched ITM forward (auto-halved on CUDA OOM)
"""
import argparse
import copy
import datetime
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.backends.cudnn as cudnn
from PIL import Image
from ruamel.yaml import YAML
from torch.utils.data import DataLoader
from torchvision import transforms

import eval_AET as E
import utils
from dataset import paired_dataset2
from SA_AET import Attacker as _BaseAttacker

PERIMG_DIR = Path(os.environ["PERIMG_DIR"]) if os.environ.get("PERIMG_DIR") else None
ADV_DIR = Path(os.environ["ADV_DIR"]) if os.environ.get("ADV_DIR") else None
SKIP_SOURCE_ITM = os.environ.get("SKIP_SOURCE_ITM", "0") == "1"
for _d in (PERIMG_DIR, ADV_DIR):
    if _d is not None:
        _d.mkdir(parents=True, exist_ok=True)

STATE = {"source_model": None, "img_names": [], "counter": 0, "adv_texts": [], "attack_time": 0.0}


# ── attacker that records adversarial outputs ───────────────────────────────────────────────
class RecordingAttacker(_BaseAttacker):
    def attack(self, imgs, *a, **kw):
        adv_imgs, adv_txts, t = super().attack(imgs, *a, **kw)
        STATE["attack_time"] += float(t)
        STATE["adv_texts"].extend(list(adv_txts))
        for k in range(adv_imgs.shape[0]):
            idx = STATE["counter"]; STATE["counter"] += 1
            if ADV_DIR is not None and idx < len(STATE["img_names"]):
                name = Path(STATE["img_names"][idx]).stem + ".png"
                transforms.ToPILImage()(adv_imgs[k].detach().float().cpu().clamp(0, 1)).save(ADV_DIR / name)
        return adv_imgs, adv_txts, t


# ── ITM re-ranking (ALBEF/TCL): batched, numerically equivalent to eval_AET.retrieval_score ──
#    The original runs one fusion forward per query (k_test pairs each). Here ITM_CHUNK queries are
#    stacked into one forward (ITM_CHUNK * k_test pairs). Halves the chunk automatically on CUDA OOM.
#    FAST_ITM=0 falls back to the original loop.
_orig_retrieval_score = E.retrieval_score
FAST_ITM = os.environ.get("FAST_ITM", "1") == "1"
ITM_CHUNK = int(os.environ.get("ITM_CHUNK", "4"))


@torch.no_grad()
def _itm_scores(model, q_embeds, q_atts, c_embeds, c_atts, sims, k, device, query_is_image, chunk):
    nq, nc = sims.shape
    out = torch.full((nq, nc), -100.0, device=device)
    s = 0
    while s < nq:
        n = min(chunk, nq - s)
        try:
            _, topk_idx = sims[s:s + n].topk(k=k, dim=1)                    # n x k
            flat = topk_idx.reshape(-1)
            if query_is_image:   # query = image, candidates = texts
                enc = q_embeds[s:s + n].to(device).repeat_interleave(k, dim=0)
                txt, att = c_embeds[flat].to(device), c_atts[flat].to(device)
            else:                # query = text, candidates = images
                enc = c_embeds[flat].to(device)
                txt = q_embeds[s:s + n].to(device).repeat_interleave(k, dim=0)
                att = q_atts[s:s + n].to(device).repeat_interleave(k, dim=0)
            enc_att = torch.ones(enc.size()[:-1], dtype=torch.long, device=device)
            o = model.text_encoder(encoder_embeds=txt, attention_mask=att, encoder_hidden_states=enc,
                                   encoder_attention_mask=enc_att, return_dict=True, mode="fusion")
            score = model.itm_head(o.last_hidden_state[:, 0, :])[:, 1].view(n, k)
            out[s:s + n].scatter_(1, topk_idx.to(device), score.to(out.dtype))
            s += n
        except torch.cuda.OutOfMemoryError:
            if chunk == 1:
                raise
            torch.cuda.empty_cache()
            chunk = max(1, chunk // 2)
            print(f"  [fast ITM] OOM -> chunk={chunk}")
    return out, chunk


@torch.no_grad()
def _fast_retrieval_score(model, image_feats, image_embeds, text_feats, text_embeds, text_atts,
                          num_image, num_text, device=None):
    device = device if device is not None else image_embeds.device
    k = E.config["k_test"]
    sims = image_feats @ text_feats.t()
    t0 = time.time()
    i2t, ch = _itm_scores(model, image_embeds, None, text_embeds, text_atts, sims, k, device, True, ITM_CHUNK)
    t2i, _ = _itm_scores(model, text_embeds, text_atts, image_embeds, None, sims.t(), k, device, False, ch)
    print(f"  [fast ITM] {num_image} img + {num_text} txt queries re-ranked (k={k}) in {time.time() - t0:.0f}s")
    return i2t, t2i


def _retrieval_score(model, image_feats, image_embeds, text_feats, text_embeds, text_atts,
                     num_image, num_text, device=None):
    if SKIP_SOURCE_ITM and model is STATE["source_model"]:
        sims = (image_feats @ text_feats.t()).to(device if device is not None else "cpu")
        return sims, sims.t()
    fn = _fast_retrieval_score if FAST_ITM else _orig_retrieval_score
    return fn(model, image_feats, image_embeds, text_feats, text_embeds, text_atts,
              num_image, num_text, device=device)


E.retrieval_score = _retrieval_score
E.Attacker = RecordingAttacker


# ── metrics (same output format as eval_AET.itm_eval, plus per-sample ranks, safe division) ──
def _asr(origin, after):
    if len(origin) == 0:
        return 0.0
    return round(100.0 * len(np.setdiff1d(origin, after)) / len(origin), 2)


def itm_eval(scores_i2t, scores_t2i, img2txt, txt2img, model_name, rank_dir):
    i2t_ranks = np.zeros(scores_i2t.shape[0], dtype=np.int64)
    for index, score in enumerate(scores_i2t):
        inds = np.argsort(score)[::-1]
        i2t_ranks[index] = min(int(np.where(inds == i)[0][0]) for i in img2txt[index])
    t2i_ranks = np.zeros(scores_t2i.shape[0], dtype=np.int64)
    for index, score in enumerate(scores_t2i):
        inds = np.argsort(score)[::-1]
        t2i_ranks[index] = int(np.where(inds == txt2img[index])[0][0])

    res = {}
    for q, ranks in (("txt", i2t_ranks), ("img", t2i_ranks)):
        tag = "tr" if q == "txt" else "ir"
        for k in (1, 5, 10):
            acc = 100.0 * len(np.where(ranks < k)[0]) / len(ranks)
            after = np.where(ranks < k)[0]
            origin = np.load(f"{rank_dir}/{model_name}_{tag}{k}_rank_index.npy")
            res[f"{q}_r{k}_ASR ({q}_r{k})"] = f"{_asr(origin, after)}({acc})"
    order = ["txt_r1_ASR (txt_r1)", "txt_r5_ASR (txt_r5)", "txt_r10_ASR (txt_r10)",
             "img_r1_ASR (img_r1)", "img_r5_ASR (img_r5)", "img_r10_ASR (img_r10)"]
    return {k: res[k] for k in order}, i2t_ranks, t2i_ranks


def eval_asr(model, ref_model, tokenizer, t_models, t_ref_models, t_tokenizers, t_test_transforms,
             data_loader, device, args, config):
    print("Start eval")
    start = time.time()
    s_i2t, s_t2i, t_i2ts, t_t2is = E.retrieval_eval(model, ref_model, t_models, t_ref_models, t_test_transforms,
                                                     data_loader, tokenizer, t_tokenizers, device, args, config)
    t_names = [m for m in args.model_list if m != args.source_model]
    ds = data_loader.dataset
    summary = {}
    with open("./result.txt", "a") as f:
        f.write("\n")
        for name, i2t, t2i in zip([args.source_model] + t_names, [s_i2t] + t_i2ts, [s_t2i] + t_t2is):
            res, r_i2t, r_t2i = itm_eval(i2t, t2i, ds.img2txt, ds.txt2img, name, args.original_rank_index_path)
            line = "Performance on {}: \n {}".format(name, res)
            f.write(line + "\n"); f.flush()
            print(line)
            summary[name] = res
            if PERIMG_DIR is not None:
                np.save(PERIMG_DIR / f"{name}_i2t_rank.npy", r_i2t)
                np.save(PERIMG_DIR / f"{name}_t2i_rank.npy", r_t2i)
    if PERIMG_DIR is not None:
        json.dump({"images": [str(x) for x in ds.image], "adv_texts": STATE["adv_texts"],
                   "clean_texts": list(ds.text), "results": summary,
                   "attack_time_s": STATE["attack_time"], "args": vars(args)},
                  open(PERIMG_DIR / "summary.json", "w"), indent=1, default=str)
    torch.cuda.empty_cache()
    print("Evaluate time {}".format(str(datetime.timedelta(seconds=int(time.time() - start)))))


def main(args, config):
    torch.cuda.set_device(args.cuda_id)
    device = torch.device("cuda")
    seed = args.seed + utils.get_rank()
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    cudnn.benchmark = True

    print("Creating Source Model")
    model, ref_model, tokenizer = E.load_model(args, args.source_model, args.source_text_encoder, device)
    STATE["source_model"] = model

    if args.ifd_mode != "none":
        from reg_attack import RegImageAttacker
        layers = [int(x) - 1 for x in str(args.ifd_layers).split(",") if x.strip()]
        if not layers:
            raise SystemExit("--ifd_layers is required when --ifd_mode != none")

        def image_attacker_factory(normalization, **kw):
            return RegImageAttacker(model, normalization, args.ifd_mode, layers, args.lam, **kw)
        E.ImageAttacker = image_attacker_factory          # used inside E.retrieval_eval
    print(f"ifd_mode={args.ifd_mode} layers={args.ifd_layers} lam={args.lam} "
          f"skip_source_itm={SKIP_SOURCE_ITM} perimg={PERIMG_DIR} adv={ADV_DIR}")

    print("Creating Target Model")
    t_models, t_ref_models, t_tokenizers = [], [], []
    t_model_names = [m for m in args.model_list if m != args.source_model]
    for name in t_model_names:
        tm, trm, tt = E.load_model(args, name, args.target_text_encoder, device)
        t_models.append(tm); t_ref_models.append(trm); t_tokenizers.append(tt)

    print("Creating dataset")
    if args.source_model in ["ALBEF", "TCL"]:
        s_tf = transforms.Compose([
            transforms.Resize((config["image_res"], config["image_res"]), interpolation=Image.BICUBIC),
            transforms.ToTensor()])
    else:
        n_px = model.visual.input_resolution
        s_tf = transforms.Compose([transforms.Resize(n_px, interpolation=Image.BICUBIC),
                                   transforms.CenterCrop(n_px), transforms.ToTensor()])
    t_tfs = []
    for name, tm in zip(t_model_names, t_models):
        if name in ["ALBEF", "TCL"]:
            t_tfs.append(transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((config["image_res"], config["image_res"]), interpolation=Image.BICUBIC),
                transforms.ToTensor()]))
        else:
            px = tm.visual.input_resolution
            t_tfs.append(transforms.Compose([transforms.Resize(px, interpolation=Image.BICUBIC),
                                             transforms.CenterCrop(px)]))

    ds = paired_dataset2(config["test_file"], s_tf, config["image_root"])
    STATE["img_names"] = list(ds.image)
    loader = DataLoader(ds, batch_size=args.batch_size, num_workers=2, collate_fn=ds.collate_fn)
    eval_asr(model, ref_model, tokenizer, t_models, t_ref_models, t_tokenizers, t_tfs, loader, device, args, config)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--config", default="./configs/Retrieval_flickr.yaml")
    p.add_argument("--seed", default=42, type=int)
    p.add_argument("--batch_size", default=8, type=int)
    p.add_argument("--cuda_id", default=0, type=int)
    p.add_argument("--model_list", default="ALBEF,TCL,CLIP_ViT,CLIP_CNN", type=str)
    p.add_argument("--source_model", default="ALBEF", type=str)
    p.add_argument("--source_text_encoder", default="bert-base-uncased", type=str)
    p.add_argument("--target_text_encoder", default="bert-base-uncased", type=str)
    p.add_argument("--albef_ckpt", default="./checkpoints/albef_flickr.pth", type=str)
    p.add_argument("--tcl_ckpt", default="./checkpoints/tcl_flickr.pth", type=str)
    p.add_argument("--original_rank_index_path", default="./std_eval_idx/flickr30k/")
    p.add_argument("--scales", type=str, default="0.5,0.75,1.25,1.5")
    p.add_argument("--ifd_mode", default="none", choices=["none", "fma"])
    p.add_argument("--ifd_layers", default="", type=str)
    p.add_argument("--lam", default=0.0, type=float)
    args = p.parse_args()
    args.model_list = [m.strip() for m in args.model_list.split(",") if m.strip()]
    assert args.source_model in args.model_list, f"{args.source_model} not in {args.model_list}"

    config = YAML(typ="safe").load(open(args.config, "r"))
    E.args, E.config = args, config          # eval_AET's helpers read these module globals
    main(args, config)
'''


# ── 0. environment setup (each step is skipped when already done in this runtime) ────────────
def setup_environment():
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
    except ImportError:
        pass
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    FOUT.mkdir(parents=True, exist_ok=True)

    if not (REPO / "SA_AET.py").exists():
        shutil.rmtree(REPO, ignore_errors=True)
        sh(f"git clone -q https://github.com/jiaxiaojunQAQ/SA-AET.git {REPO}")

    for f in ("eval_AET.py", "eval.py"):
        p = REPO / f
        s = p.read_text()
        s2 = re.sub(r"^from dataset import paired_dataset\s*$", "from dataset import paired_dataset, paired_dataset2",
                    s, flags=re.M)
        if s2 != s:
            p.write_text(s2)

    _p = REPO / "SA_AET.py"; _s = _p.read_text()
    _OLD = """            all_texts_input = self.txt_attacker.tokenizer(all_texts, padding='max_length', truncation=True,
                                                     max_length=max_length, return_tensors="pt").to(device)
            all_texts_output = self.model.inference_text(all_texts_input)
            all_txt_supervisions = all_texts_output['text_feat']
"""
    _NEW = """            if getattr(self, "_all_txt_key", None) != (id(all_texts), len(all_texts)):   # PATCHED: encode once, chunked
                _feats = []
                for _i in range(0, len(all_texts), 256):
                    _inp = self.txt_attacker.tokenizer(all_texts[_i:_i + 256], padding='max_length', truncation=True,
                                                       max_length=max_length, return_tensors="pt").to(device)
                    _feats.append(self.model.inference_text(_inp)['text_feat'])
                self._all_txt_cache = torch.cat(_feats)
                self._all_txt_key = (id(all_texts), len(all_texts))
            all_txt_supervisions = self._all_txt_cache
"""
    if "PATCHED: encode once" not in _s:
        if _OLD in _s:
            _p.write_text(_s.replace(_OLD, _NEW))
        else:
            print("  WARNING: SA_AET.py text-encoding patch did not apply (upstream code changed?)")

    if not Path(ENV_PY).exists():
        sh(f"{sys.executable} -m pip install -q uv")
        sh("uv venv /content/sa_env --python 3.10 -q")
        sh(f"uv pip install -q --python {ENV_PY} torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121")
        sh(f"uv pip install -q --python {ENV_PY} transformers==4.28.1 timm==0.6.13 ftfy==6.1.1 "
           f"ruamel.yaml==0.17.21 'numpy<2' regex tqdm pillow setuptools==69.5.1 huggingface_hub==0.16.4")

    clip_cache = Path.home() / ".cache/clip"; clip_cache.mkdir(parents=True, exist_ok=True)
    CLIP_FILES = {"ViT-B-16.pt": "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f",
                  "RN101.pt":    "8fa8567bab74a42d41c5915025a8e4538c3bdbe8804a470a72f30b0d94fab599"}
    for fn, sha in CLIP_FILES.items():
        dst = clip_cache / fn
        ok = dst.exists() and hashlib.sha256(dst.read_bytes()).hexdigest() == sha
        for host in ("https://openaipublic.azureedge.net", "https://openaipublic.blob.core.windows.net"):
            if ok:
                break
            try:
                urllib.request.urlretrieve(f"{host}/clip/models/{sha}/{fn}", dst)
                ok = hashlib.sha256(dst.read_bytes()).hexdigest() == sha
            except Exception:
                pass
        if not ok:
            print(f"  WARNING: could not pre-download CLIP {fn}; clip.load will try itself")

    (REPO / "reg_attack.py").write_text(REG_ATTACK)
    (REPO / "eval_reg.py").write_text(EVAL_REG)
    assert (REPO / "eval_reg.py").exists() and (REPO / "reg_attack.py").exists()
    print("  wrote eval_reg.py + reg_attack.py (FMA-v4: MAD z-score band-pass + stride-2 pyramid pooling)")


# ── A. dataset split, clean rank indices, images, checkpoints, config ───────────────────────
def prepare_data():
    """Returns (SUB, run_config_path, clean_rank_index_dir)."""
    ann_full = json.load(open(REPO / "data_annotation/flickr30k_test.json"))
    assert 1 <= N_IMAGES <= len(ann_full), f"N_IMAGES must be between 1 and {len(ann_full)}"
    ann_all = ann_full[:N_IMAGES]
    sub = "full" if N_IMAGES == len(ann_full) else f"sub{N_IMAGES}"
    if sub != "full":
        print(f"  NOTE: N_IMAGES={N_IMAGES} -> subset '{sub}'. These numbers are NOT comparable to Table III.")

    ann_sub_path = f"./data_annotation/flickr30k_{sub}.json"
    json.dump(ann_all, open(REPO / ann_sub_path, "w"))

    idx_full_dir = REPO / "std_eval_idx/flickr30k"
    if sub == "full":
        idx_dir = "./std_eval_idx/flickr30k/"
    else:
        idx_dir = f"./std_eval_idx/flickr30k_{sub}/"
        sub_idx_dir = REPO / idx_dir
        sub_idx_dir.mkdir(parents=True, exist_ok=True)
        n_txt_limit = sum(len(a["caption"]) for a in ann_all)
        for npy_file in idx_full_dir.glob("*.npy"):
            arr = np.load(npy_file)
            arr_sub = arr[arr < N_IMAGES] if "_tr" in npy_file.name else arr[arr < n_txt_limit]
            np.save(sub_idx_dir / npy_file.name, arr_sub)
        print(f"  created filtered clean rank indices in {idx_dir} for {N_IMAGES} images ({n_txt_limit} captions)")

    need = sorted({a["image"].split("/")[-1] for a in ann_all})
    img_dir = LOCAL_ROOT / "flickr30k-images"; img_dir.mkdir(parents=True, exist_ok=True)
    FLICKR_CACHE.mkdir(parents=True, exist_ok=True)

    def _find_drive_copy(name):
        for d in (FLICKR_ROOT / "flickr30k-images", FLICKR_ROOT, FLICKR_CACHE):
            if (d / name).exists():
                return d / name
        return None

    missing = []
    for n in need:
        dst = img_dir / n
        if dst.exists():
            continue
        srcf = _find_drive_copy(n)
        if srcf is not None:
            shutil.copy(srcf, dst)
        else:
            missing.append(n)

    if missing:
        print(f"  extracting {len(missing)} needed images from Flickr30K zip...")
        if not FLICKR_ZIP.exists() or FLICKR_ZIP.stat().st_size < 4_000_000_000:
            sh(f"wget -q -c -O {FLICKR_ZIP} '{FLICKR_ZIP_URL}'")
        want = set(missing)
        with zipfile.ZipFile(FLICKR_ZIP) as zf:
            infos = [i for i in zf.infolist() if i.filename.endswith(".jpg") and not i.filename.startswith("__MACOSX")]
            by_name = {i.filename.split("/")[-1]: i for i in infos}
            for base in list(want):
                if base in by_name:
                    data = zf.read(by_name[base])
                    (img_dir / base).write_bytes(data)
                    (FLICKR_CACHE / base).write_bytes(data)       # cached on Drive for the next runtime
                    want.discard(base)
        if want:
            raise SystemExit(f"{len(want)} images not found in the Flickr30K zip, e.g. {sorted(want)[:3]}")
        FLICKR_ZIP.unlink(missing_ok=True)
    print(f"  Flickr30K images ready for {sub}: {len(need)} images")

    ck = REPO / "checkpoints"; ck.mkdir(exist_ok=True)
    for name in ("albef_flickr.pth", "tcl_flickr.pth"):
        if not (ck / name).exists() or (ck / name).stat().st_size < 10_000_000:
            sh(f"wget -q -O {ck / name} https://huggingface.co/Sensen02/VLPTransferAttackCheckpoints/resolve/main/{name}")

    k_test_val = min(128, N_IMAGES)       # ALBEF/TCL top-k re-ranking can never exceed the image count
    y = (REPO / "configs/Retrieval_flickr.yaml").read_text()
    y = re.sub(r"^image_root:.*$", f"image_root: '{LOCAL_ROOT}/'", y, flags=re.M)
    y = re.sub(r"^test_file:.*$", f"test_file: '{ann_sub_path}'", y, flags=re.M)
    y = re.sub(r"^k_test:.*$", f"k_test: {k_test_val}", y, flags=re.M)
    run_cfg = f"./configs/flickr_{sub}.yaml"
    (REPO / run_cfg).write_text(y)
    return sub, run_cfg, idx_dir


# ── B. job plan ─────────────────────────────────────────────────────────────────────────────
def build_plan(stage):
    """Ordered list of (source, label, mode, seed). Seeds are interleaved so ours and the baseline
    always have the same seeds finished (paired comparison) even if the runtime dies mid-plan."""
    jobs = [("CLIP_ViT", SAAET_LABEL, "none", 42)]
    if stage >= 2:
        for s in SEEDS:
            jobs += [("CLIP_ViT", FMA_LABEL, "fma", s), ("CLIP_ViT", SAAET_LABEL, "none", s)]
        if RUN_ABLATIONS:
            jobs += [("CLIP_ViT", lab, "fma", 42) for lab in ABLATIONS]
    if stage >= 3:
        for src in STAGE3_SOURCES:
            for s in STAGE3_SEEDS:
                jobs.append((src, SAAET_LABEL, "none", s))
                if src in VIT_SOURCES:
                    jobs.append((src, FMA_LABEL, "fma", s))
    seen, out = set(), []
    for j in jobs:
        if j not in seen:
            seen.add(j); out.append(j)
    return out


def src_itm_flag(src):
    """Whether this source's own column is ITM re-ranked (None for CLIP: no ITM head)."""
    return bool(SOURCE_ITM) if src in ITM_MODELS else None


# ── C. result parsing & cache ───────────────────────────────────────────────────────────────
def parse(block):
    out = {}
    for name, body in re.findall(r"Performance on (\w+): \n (\{.*?\})", block):
        d = eval(body)
        out[name] = (float(re.findall(r"[\d.]+", d["txt_r1_ASR (txt_r1)"])[0]),
                     float(re.findall(r"[\d.]+", d["img_r1_ASR (img_r1)"])[0]))
    return out


def parse_full(block):
    out = {}
    for name, body in re.findall(r"Performance on (\w+): \n (\{.*?\})", block):
        d, m = eval(body), {"asr": {}, "acc": {}}
        for q, qq in (("TR", "txt"), ("IR", "img")):
            for k in (1, 5, 10):
                v = re.findall(r"[\d.]+", d[f"{qq}_r{k}_ASR ({qq}_r{k})"])
                m["asr"][f"{q}@{k}"], m["acc"][f"{q}@{k}"] = float(v[0]), float(v[1])
        out[name] = m
    return out


def load_saved(src, sub):
    """{(label, seed): {target: (TR, IR)}} for finished runs of this source/subset/ITM setting."""
    R = {}
    for f in sorted(glob.glob(str(FOUT / f"comparison_FLICKR-*_{src}_{sub}_*.json"))):
        try:
            d = json.load(open(f))
        except Exception:
            continue
        if d.get("subset") != sub or d.get("source", src) != src:
            continue
        if src in ITM_MODELS and bool(d.get("src_itm", False)) != bool(SOURCE_ITM):
            continue                                    # older runs were made with SKIP_SOURCE_ITM=1
        if not all(m in d.get("asr", {}) for m in MODELS):
            continue                                    # incomplete / broken run: never reuse
        R[(d["label"], d["seed"])] = {k: tuple(v) for k, v in d["asr"].items()}
    return R


def run_job(src, label, mode, seed, sub, run_cfg, idx_dir):
    if mode == "fma" and src not in VIT_SOURCES:
        raise SystemExit(f"FMA needs a ViT source; {src} has none (skip it, report N/A)")
    tag = f"FLICKR-{label}_s{seed}_{src}_{sub}_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
    res_file = REPO / "result.txt"
    before = res_file.read_text() if res_file.exists() else ""
    extra = f"--ifd_layers {LAYERS} --lam {LAM}" if mode == "fma" else ""
    perimg_local, adv_local = Path(f"/content/perimg_{tag}"), Path(f"/content/adv_{tag}")
    for d_ in (perimg_local, adv_local):
        shutil.rmtree(d_, ignore_errors=True); d_.mkdir(parents=True)
    skip_itm = "0" if src_itm_flag(src) else "1"
    env = f"PERIMG_DIR={perimg_local} SKIP_SOURCE_ITM={skip_itm} FAST_ITM=1 ITM_CHUNK=16 "   # auto-halves on OOM
    env += f"ADV_DIR={adv_local} " if SAVE_IMAGES else ""
    if mode == "fma":
        env += f"FMA_CFG='{json.dumps(LABEL_CFG.get(label, FMA_CFG))}' "
    print(f"\n  -> running source={src} {label} seed={seed} on {sub} ({N_IMAGES} images)")
    sh(f"cd {REPO} && {env}{ENV_PY} eval_reg.py --config {run_cfg} --source_model {src} "
       f"--batch_size {BATCH_SIZE} --albef_ckpt ./checkpoints/albef_flickr.pth --tcl_ckpt ./checkpoints/tcl_flickr.pth "
       f"--original_rank_index_path {idx_dir} --cuda_id 0 --seed {seed} --ifd_mode {mode} {extra}",
       log=str(FOUT / f"log_{tag}.txt"))
    new = res_file.read_text()[len(before):]
    (FOUT / f"result_{tag}.txt").write_text(new)
    asr = parse(new)
    if not all(m in asr for m in MODELS):
        raise SystemExit(f"run finished but result.txt has only {sorted(asr)}; see log_{tag}.txt (not cached)")
    perimg_drive = FOUT / "perimg" / tag
    shutil.copytree(perimg_local, perimg_drive, dirs_exist_ok=True)
    if SAVE_IMAGES:
        shutil.copytree(adv_local, FOUT / "adv" / tag, dirs_exist_ok=True)
    attack_s = None
    try:
        attack_s = json.load(open(perimg_local / "summary.json")).get("attack_time_s")
    except Exception:
        pass
    json.dump({"label": label, "mode": mode, "layers": LAYERS if mode == "fma" else "",
               "lam": LAM if mode == "fma" else 0.0, "seed": seed,
               "cfg": LABEL_CFG.get(label, FMA_CFG) if mode == "fma" else None,
               "dataset": "flickr30k", "source": src, "subset": sub, "n_images": N_IMAGES,
               "batch_size": BATCH_SIZE, "src_itm": src_itm_flag(src), "attack_time_s": attack_s,
               "asr": asr, "full": parse_full(new), "perimg_dir": str(perimg_drive),
               "clean_rank_dir": str(REPO / idx_dir)},
              open(FOUT / f"comparison_{tag}.json", "w"), indent=2)
    if attack_s:
        print(f"  attack time: {attack_s / 60:.1f} min")


# ── D. reporting ────────────────────────────────────────────────────────────────────────────
def agg(R, label, seeds=None):
    """Mean/std per target cell over the seeds available for this label.
    Returns (n_seeds, {target: (TR_mean, TR_std, IR_mean, IR_std)}, seeds_used)."""
    runs = {s: r for (lab, s), r in R.items() if lab == label and (seeds is None or s in seeds)}
    if not runs:
        return 0, {}, []
    out = {}
    for t in MODELS:
        tr = np.array([runs[s][t][0] for s in runs]); ir = np.array([runs[s][t][1] for s in runs])
        sd = lambda x: float(x.std(ddof=1)) if len(x) > 1 else 0.0
        out[t] = (float(tr.mean()), sd(tr), float(ir.mean()), sd(ir))
    return len(runs), out, sorted(runs)


def transfer_mean(cells, src):
    T = [t for t in MODELS if t != src]
    return sum(cells[t][0] + cells[t][2] for t in T) / (2 * len(T))


def fmt(m, s, n):
    return f"{m:.2f}" if n < 2 else f"{m:.2f}±{s:.2f}"


def repro_check(src, R):
    """Stage-1 gate: reproduced SA-AET (seed 42) vs the paper's Table III row."""
    if (SAAET_LABEL, 42) not in R:
        return None
    r, p = R[(SAAET_LABEL, 42)], PAPER_T3[src]["SA-AET"]
    print("\n" + "=" * 100)
    print(f"  STAGE-1 REPRODUCTION CHECK — SA-AET, source {src}, seed 42, Flickr30K full, R@1 ASR (%)")
    print("=" * 100)
    print(f"  {'target':<10}{'metric':<8}{'paper':>9}{'ours':>9}{'Δ':>9}   status")
    worst = 0.0
    for t in MODELS:
        if t == src:
            continue
        for k, mname in ((0, "TR@1"), (1, "IR@1")):
            d = r[t][k] - p[t][k]
            worst = max(worst, abs(d))
            st = "ok" if abs(d) <= REPRO_TOL else ("check" if abs(d) <= 2 * REPRO_TOL else "MISMATCH")
            print(f"  {t:<10}{mname:<8}{p[t][k]:>9.2f}{r[t][k]:>9.2f}{d:>+9.2f}   {st}")
    wb = r[src]
    print(f"  white-box {src}: TR {wb[0]:.2f} / IR {wb[1]:.2f} (paper {p[src][0]:.2f} / {p[src][1]:.2f})")
    if worst <= REPRO_TOL:
        verdict = f"REPRODUCED (all transfer cells within ±{REPRO_TOL:g}). Go to STAGE = 2."
    elif worst <= 2 * REPRO_TOL:
        verdict = (f"CLOSE (worst |Δ| = {worst:.2f}). Usually seed noise; acceptable, but report your own "
                   f"reproduced SA-AET numbers in the table, not the paper's.")
    else:
        verdict = (f"NOT REPRODUCED (worst |Δ| = {worst:.2f}). Debug before stage 2: check N_IMAGES=1000, "
                   f"checkpoints, clean rank dir, batch size {BATCH_SIZE}.")
    print("  -> " + verdict)
    print("=" * 100)
    return worst


def table3_report(results, sub):
    """results: {source: R}. Prints the Table III draft and saves Markdown + CSV to Drive."""
    cols = MODELS
    lines, rows = [], []
    head = "| Source | Attack | " + " | ".join(f"{t} TR@1 | {t} IR@1" for t in cols) + " | Transfer mean |"
    lines += [f"### Table III draft — Flickr30K ({sub}, {N_IMAGES} images), R@1 ASR (%)", "",
              head, "|" + "---|" * (2 + 2 * len(cols) + 1)]

    def add(src, name, cells, n):
        tm = transfer_mean(cells, src)
        vals = []
        for t in cols:
            a, b = fmt(cells[t][0], cells[t][1], n), fmt(cells[t][2], cells[t][3], n)
            if t == src:
                a, b = f"*{a}*", f"*{b}*"           # white-box cell
            vals += [a, b]
        lines.append(f"| {src} | {name} | " + " | ".join(vals) + f" | {tm:.2f} |")
        rows.append([src, name, n] + [x for t in cols for x in (cells[t][0], cells[t][1], cells[t][2], cells[t][3])]
                    + [tm])
        return tm

    for src, R in results.items():
        paper = {t: (v[0], 0.0, v[1], 0.0) for t, v in PAPER_T3[src]["SA-AET"].items()}
        add(src, "SA-AET (paper)", paper, 1)
        n_b, base, seeds_b = agg(R, SAAET_LABEL)
        n_f, ours, seeds_f = agg(R, FMA_LABEL)
        base_tm = add(src, f"SA-AET (repro, n={n_b})", base, n_b) if n_b else None
        if src not in VIT_SOURCES:
            lines.append(f"| {src} | FMA-v4 (ours) | " + " | ".join(["N/A"] * (2 * len(cols)))
                         + " | N/A (needs ViT source) |")
        elif n_f:
            ours_tm = add(src, f"FMA-v4 (ours, n={n_f})", ours, n_f)
            common = sorted(set(seeds_b) & set(seeds_f))
            if common:
                _, b_c, _ = agg(R, SAAET_LABEL, common)
                _, o_c, _ = agg(R, FMA_LABEL, common)
                d = {t: (o_c[t][0] - b_c[t][0], 0.0, o_c[t][2] - b_c[t][2], 0.0) for t in cols}
                per_seed = [transfer_mean({t: (R[(FMA_LABEL, s)][t][0], 0, R[(FMA_LABEL, s)][t][1], 0) for t in cols}, src)
                            - transfer_mean({t: (R[(SAAET_LABEL, s)][t][0], 0, R[(SAAET_LABEL, s)][t][1], 0) for t in cols}, src)
                            for s in common]
                vals = []
                for t in cols:
                    vals += [f"{d[t][0]:+.2f}", f"{d[t][2]:+.2f}"]
                wins = sum(x > 0 for x in per_seed)
                lines.append(f"| {src} | Δ ours − repro (seeds {common}) | " + " | ".join(vals)
                             + f" | {transfer_mean(d, src):+.2f} (better on {wins}/{len(common)} seeds) |")
        lines.append("|" + " |" * (2 + 2 * len(cols) + 1))
    lines += ["", "*italic* = white-box (source = target). Values with ± are mean ± std over seeds.",
              f"Source ITM re-ranking for ALBEF/TCL sources: {'on (paper setting)' if SOURCE_ITM else 'OFF (white-box cell not paper-comparable)'}.",
              "FMA-v4 hyper-parameters frozen from the CLIP_ViT sub400 study (λ=4, layers 7–10)."]
    md = "\n".join(lines)
    print("\n" + md + "\n")
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    (FOUT / f"table3_flickr_{sub}_{stamp}.md").write_text(md)
    with open(FOUT / f"table3_flickr_{sub}_{stamp}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["source", "attack", "n_seeds"]
                   + [f"{t}_{m}" for t in cols for m in ("TR_mean", "TR_std", "IR_mean", "IR_std")]
                   + ["transfer_mean"])
        w.writerows(rows)
    print(f"  saved {FOUT}/table3_flickr_{sub}_{stamp}.md and .csv")


def ablation_report(R, src, sub):
    """Optional (RUN_ABLATIONS): single-seed ablation table for one source on this subset."""
    T = [t for t in MODELS if t != src]
    mean = lambda r: sum(r[t][k] for t in T for k in (0, 1)) / (2 * len(T))
    have = [lab for lab in ABLATIONS if (lab, 42) in R]
    if (FMA_LABEL, 42) not in R or not have:
        print("  ablation table skipped: full FMA-v4 run or ablation runs missing")
        return
    fm = mean(R[(FMA_LABEL, 42)])
    base_m = mean(R[(SAAET_LABEL, 42)]) if (SAAET_LABEL, 42) in R else 0.0
    print("\n" + "=" * 118)
    print(f"  FMA-v4 ABLATION — Flickr30K ({N_IMAGES} images, '{sub}'), Source: {src}, seed 42, R@1 ASR (%) TR / IR")
    print("=" * 118)
    print("  " + f"{'Variant':<26}{'Configuration / Removed':<44}"
          + "".join(f"{t:>15}" for t in T) + f"{'Mean':>8}{'vs Base':>9}{'Cost':>8}")
    print("  " + "-" * 116)
    rows = [(DISPLAY[FMA_LABEL], "Full FMA-v4 (z-BandPass + Stride2-MS + Focal)", FMA_LABEL)] \
        + [(ABLATIONS[lab][0], ABLATIONS[lab][1], lab) for lab in have] \
        + [(DISPLAY[SAAET_LABEL], "No intermediate feature loss", SAAET_LABEL)]
    for name, desc, lab in rows:
        if (lab, 42) not in R:
            continue
        r = R[(lab, 42)]
        m_val = mean(r)
        vs_base = f"{m_val - base_m:>+8.2f}" if (SAAET_LABEL, 42) in R and lab != SAAET_LABEL else "    —   "
        cost = "   —    " if lab == FMA_LABEL else f"{fm - m_val:>+8.2f}"
        print("  " + f"{name:<26}{desc[:42]:<44}"
              + "".join(f"{r[t][0]:>6.2f}/{r[t][1]:>5.2f}" for t in T)
              + f"{m_val:>8.2f}{vs_base:>9}{cost:>8}")
    print("=" * 118)


# ── E. main ─────────────────────────────────────────────────────────────────────────────────
def main():
    setup_environment()
    sub, run_cfg, idx_dir = prepare_data()
    plan = build_plan(STAGE)
    sources = list(dict.fromkeys(s for s, *_ in plan))

    print("\n" + "=" * 100)
    print(f"  STAGE {STAGE} PLAN — Flickr30K '{sub}', {len(plan)} jobs "
          f"(source ITM for ALBEF/TCL: {'on' if SOURCE_ITM else 'off'}, batch {BATCH_SIZE})")
    print("=" * 100)
    cache = {s: load_saved(s, sub) for s in sources}
    todo = []
    for src, label, mode, seed in plan:
        done = (label, seed) in cache[src]
        print(f"  [{'CACHED' if done else 'todo  '}] source={src:<9} {DISPLAY.get(label, label):<22} seed={seed}")
        if not done:
            todo.append((src, label, mode, seed))
    print(f"  -> {len(todo)} run(s) to do")

    failures = []
    for i, (src, label, mode, seed) in enumerate(todo, 1):
        print(f"\n  ── job {i}/{len(todo)} ──")
        try:
            run_job(src, label, mode, seed, sub, run_cfg, idx_dir)
        except SystemExit as e:
            print(f"  !! FAILED: source={src} {label} seed={seed}: {e}")
            failures.append((src, label, seed, str(e)))

    results = {s: load_saved(s, sub) for s in sources}
    if sub == "full":
        for src in sources:
            if src == "CLIP_ViT" or STAGE >= 3:
                repro_check(src, results[src])
    table3_report(results, sub)
    if RUN_ABLATIONS:
        ablation_report(results["CLIP_ViT"], "CLIP_ViT", sub)
    if failures:
        print("\n  FAILED RUNS (not cached — fix and re-run this cell; finished runs are reused):")
        for src, label, seed, msg in failures:
            print(f"   - source={src} {label} seed={seed}: {msg}")
    print(f"\n  STAGE {STAGE} finished: {len(todo) - len(failures)}/{len(todo)} new run(s) succeeded.")


if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
Start eval
Computing features for evaluation adv...
  RegImageAttacker: mode=fma, layers=[7, 8, 9, 10], lam=4.0, cfg={'act_power': 1.0, 'reg_z': 3.0, 'bg_z': -0.8, 'patch2x2_beta': 0.25, 'patch4x4_beta': 0.2, 'pool_stride': 2, 'cls_beta': 0.0, 'focal_alpha': 2.5, 'focal_margin': 0.15, 'layer_focus': 0.8}
Prepare memory
[0.5, 0.75, 1.25, 1.5]
Forward
--------------------> batch:0/125
--------------------> batch:1/125
--------------------> batch:2/125
--------------------> batch:3/125
--------------------> batch:4/125
--------------------> batch:5/125
--------------------> batch:6/125
--------------------> batch:7/125
--------------------> batch:8/125
--------------------> batch:9/125
--------------------> batch:10/125
--------------------> batch:11/125
--------------------> batch:12/125
--------------------> batch:13/125
--------------------> batch:14/125
--------------------> batch:15/125
--------------------> batch:16/125
------------